In [0]:
%sql
-- Create 'trend_market_project' folder
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.trend_market_project;

PART 1: Initialize Paths and Ingest Data

In [0]:
# Step 1: Define paths
trips_path = "/Volumes/msbabigdata/spark/trend_market_project/yellow_tripdata_2023_sample.parquet"
zone_path = "/Volumes/msbabigdata/spark/trend_market_project/Taxi_zone_lookup.csv"

In [0]:
# Step 2: Ingest data
df_raw = spark.read.parquet(trips_path)
zone_df = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_path)

In [0]:
# Step 3: Standardize column names to lowercase immediately (fixes the 'Airport_fee' issue)
df_trips = df_raw.toDF(*[c.lower() for c in df_raw.columns])

In [0]:
# --- VALIDATION OF PART 1: Profile the raw data ---
print(f"Raw record count: {df_trips.count()}")
df_trips.select("tpep_pickup_datetime", "fare_amount", "trip_distance").summary().show()

Raw record count: 3000
+-------+------------------+-----------------+
|summary|       fare_amount|    trip_distance|
+-------+------------------+-----------------+
|  count|              3000|             3000|
|   mean|19.197499999999984| 3.85141333333333|
| stddev|19.531142412174486|4.717278570128284|
|    min|             -80.0|              0.0|
|    25%|               8.6|             1.21|
|    50%|              12.8|              2.2|
|    75%|              22.6|              4.4|
|    max|             400.0|            75.57|
+-------+------------------+-----------------+



PART 2: Build the Cleaning Pipeline

In [0]:
from pyspark.sql import functions as F

In [0]:
def clean_taxi_data(df):
    return (df
        # 1. Handle Erroneous Dates: Filter strictly for 2023
        .filter(F.year("tpep_pickup_datetime") == 2023)
        
        # 2. Handle Null/Invalid Zone IDs (NYC zones are 1-263)
        .filter(F.col("pulocationid").isNotNull() & (F.col("pulocationid").between(1, 263)))
        .filter(F.col("dolocationid").isNotNull() & (F.col("dolocationid").between(1, 263)))
        
        # 3. Handle Outliers: Fare >= $2.50 (NYC min) and sensible distance
        .filter((F.col("fare_amount") >= 2.5) & (F.col("fare_amount") < 500))
        .filter((F.col("trip_distance") > 0) & (F.col("trip_distance") < 100))
        
        # 4. Remove Duplicate records
        .dropDuplicates()
        
        # 5. Timestamp formatting & Timezone Normalization
        # We explicitly convert the UTC data to New York time
        .withColumn("tpep_pickup_datetime", F.from_utc_timestamp(F.col("tpep_pickup_datetime"), "America/New_York"))
        .withColumn("tpep_dropoff_datetime", F.from_utc_timestamp(F.col("tpep_dropoff_datetime"), "America/New_York"))
        
        # Add a date column for partitioning later
        .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    )

In [0]:
df_cleaned = clean_taxi_data(df_trips)

In [0]:
# --- VALIDATION of PART 2: Check Cleaning Results ---
print(f"Cleaned record count: {df_cleaned.count()}")

Cleaned record count: 2852


In [0]:
# Check for remaining nulls in critical columns
df_cleaned.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["pulocationid", "fare_amount"]]).show()

+------------+-----------+
|pulocationid|fare_amount|
+------------+-----------+
|           0|          0|
+------------+-----------+



In [0]:
# Verify Date Range (Should only be 2023)
df_cleaned.select(F.min("tpep_pickup_datetime"), F.max("tpep_pickup_datetime")).show()

+-------------------------+-------------------------+
|min(tpep_pickup_datetime)|max(tpep_pickup_datetime)|
+-------------------------+-------------------------+
|      2022-12-31 19:00:47|      2023-02-28 20:03:01|
+-------------------------+-------------------------+



PART 3: Join with Zone Lookup

In [0]:
# Join cleaned trips with the lookup table
# Use left join to ensure we keep trip records even if a zone ID is weird

# Join 1: Attach names for the Pickup Location
final_df = df_cleaned.join(
    zone_df.select(F.col("LocationID").alias("puloc_id"), 
                   F.col("Borough").alias("pickup_borough"), 
                   F.col("Zone").alias("pickup_zone")), 
    df_cleaned.pulocationid == F.col("puloc_id"), 
    "left"
).drop("puloc_id")

In [0]:
# Join 2: Attach names for the Dropoff Location (Requirement B)
final_df = final_df.join(
    zone_df.select(F.col("LocationID").alias("doloc_id"), 
                   F.col("Borough").alias("dropoff_borough"), 
                   F.col("Zone").alias("dropoff_zone")), 
    final_df.dolocationid == F.col("doloc_id"), 
    "left"
).drop("doloc_id")

In [0]:
# --- VALIDATION of PART 3: Check Join ---
# Ensure we have borough names instead of just IDs
final_df.select("pulocationid", "pickup_borough", "pickup_zone").distinct().limit(5).show()

+------------+--------------+-------------------+
|pulocationid|pickup_borough|        pickup_zone|
+------------+--------------+-------------------+
|          79|     Manhattan|       East Village|
|         144|     Manhattan|Little Italy/NoLiTa|
|          75|     Manhattan|  East Harlem South|
|         194|     Manhattan|    Randalls Island|
|         140|     Manhattan|    Lenox Hill East|
+------------+--------------+-------------------+



In [0]:
# --- VALIDATION of PART 3: check both ends of the trip
final_df.select(
    "pulocationid", "pickup_borough", 
    "dolocationid", "dropoff_borough"
).distinct().limit(5).show()

+------------+--------------+------------+---------------+
|pulocationid|pickup_borough|dolocationid|dropoff_borough|
+------------+--------------+------------+---------------+
|          43|     Manhattan|         237|      Manhattan|
|         164|     Manhattan|         236|      Manhattan|
|         164|     Manhattan|         232|      Manhattan|
|          13|     Manhattan|         211|      Manhattan|
|         239|     Manhattan|         133|       Brooklyn|
+------------+--------------+------------+---------------+



PART 4: Write to Delta Lake

In [0]:
target_table = "msbabigdata.spark.trend_market_cleaned_trips"

(final_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .partitionBy("pickup_date", "pulocationid")
 .saveAsTable(target_table))

In [0]:
# --- VALIDATION of PART 4: Query the Delta Table ---
print("Final Table Sample:")
display(spark.sql(f"SELECT * FROM {target_table} LIMIT 10"))

Final Table Sample:


vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_date,pickup_borough,pickup_zone,dropoff_borough,dropoff_zone
2,2022-12-31T19:52:15.000Z,2022-12-31T20:01:38.000Z,1.0,1.69,1.0,N,231,158,1,11.4,1.0,0.5,2.0,0.0,1.0,18.4,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Meatpacking/West Village West
2,2022-12-31T19:44:16.000Z,2022-12-31T19:55:47.000Z,2.0,2.15,1.0,N,231,164,2,13.5,1.0,0.5,0.0,0.0,1.0,18.5,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown South
2,2022-12-31T19:23:15.000Z,2022-12-31T19:59:20.000Z,1.0,6.14,1.0,N,231,238,1,38.7,1.0,0.5,8.74,0.0,1.0,52.44,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Upper West Side North
2,2022-12-31T19:32:17.000Z,2022-12-31T20:25:08.000Z,1.0,9.43,1.0,Y,231,116,1,56.9,1.0,0.5,0.0,0.0,1.0,61.9,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Hamilton Heights
1,2022-12-31T19:45:24.000Z,2022-12-31T20:07:35.000Z,3.0,3.7,1.0,N,231,161,2,21.2,3.5,0.5,0.0,0.0,1.0,26.2,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown Center
1,2022-12-31T19:54:44.000Z,2022-12-31T20:13:06.000Z,1.0,5.9,1.0,N,231,161,3,26.1,3.5,0.5,0.0,0.0,1.0,31.1,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown Center
1,2022-12-31T19:48:00.000Z,2022-12-31T20:28:58.000Z,1.0,5.8,1.0,N,231,239,1,38.7,3.5,0.5,6.56,0.0,1.0,50.26,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Upper West Side South
1,2022-12-31T19:41:30.000Z,2022-12-31T20:03:23.000Z,2.0,2.8,1.0,N,231,164,1,22.6,3.5,0.5,5.0,0.0,1.0,32.6,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Midtown South
1,2022-12-31T19:14:15.000Z,2022-12-31T19:22:39.000Z,0.0,1.6,1.0,N,231,114,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Greenwich Village South
1,2022-12-31T19:18:42.000Z,2022-12-31T19:34:09.000Z,1.0,2.1,1.0,N,231,107,1,15.6,3.5,0.5,4.1,0.0,1.0,24.7,2.5,0.0,2022-12-31,Manhattan,TriBeCa/Civic Center,Manhattan,Gramercy
